# Predict the cell cycle phase score from phase contrast (OPS0077 NTC)

This notebook runs `scripts/ops0077_phase_predict_cc_phase.py`. It asks how well **label-free phase-contrast features** predict the cell cycle phase score `cc_phase` that was derived independently from the stains (p21 / Rb / pRb and DNA content).

**Design**
- **Label.** `cc_phase` from `scripts/ops0077_cell_cycle_score.py` (circular, in [0, 1)). Phase contrast is *not* used to build it.
- **Target.** Circular regression: the model predicts `(sin 2π·phase, cos 2π·phase)` and the predicted phase is the angle of that pair, so 0 and 1 are the same point.
- **Predictors.** Phase-image features only (OrganelleProfiler `phase2d` / `focus3d` structures and, for a subset of cells, CellProfiler texture and intensity). Fluorescence-derived features (`gfp`, `mcherry`) are always excluded, and mask-geometry features (cell / nucleus shape from the 4i segmentation) are excluded by default.
- **Models.** Ridge regression on quantile-normalised features, and gradient boosting on 100 PCs.
- **Validation.** Out-of-fold, grouped so neighbouring cells never straddle train and test: 5-fold over tiles (primary) and leave-one-well-out (stringent). Baselines using only cell size, or only registration distance and local cell number, show that the model learns more than those confounds.

**Inputs** (relative to the repository root)

| file | content |
|---|---|
| `data/altair/OPS0077_NTC_phase_OrganelleProfiler.h5ad` | single-cell phase-contrast features |
| `reports/ops0077_cell_cycle_score/cell_cycle_scores.csv` | `cc_phase` labels. Create it first with `15_run_ops0077_cell_cycle_score_markers_dna.ipynb` |

**Outputs** go to `reports/ops0077_phase_predicts_cc_phase/`. The full run takes roughly 10 to 20 minutes.

## 1. Requirements

In [ ]:
# Run once if needed (uncomment). Python >= 3.10.
# %pip install anndata numpy pandas scipy scikit-learn matplotlib

## 2. Configuration and input checks

In [ ]:
import subprocess, sys, json
from pathlib import Path

ROOT = Path("..").resolve() if (Path("..") / "scripts").exists() else Path(".").resolve()
SCRIPT = ROOT / "scripts/ops0077_phase_predict_cc_phase.py"
INPUTS = {
    "phase-contrast features": ROOT / "data/altair/OPS0077_NTC_phase_OrganelleProfiler.h5ad",
    "cc_phase labels": ROOT / "reports/ops0077_cell_cycle_score/cell_cycle_scores.csv",
}
OUT = ROOT / "reports/ops0077_phase_predicts_cc_phase"

# options passed to the script
QUICK = False                    # True: 25% of the cells and no leave-one-well-out/ablation (a few minutes, for testing)
INCLUDE_MASK_MORPHOLOGY = False  # True: also use cell / nucleus mask geometry (taken from the 4i segmentation, not phase images)
MAX_REG_DIST = None              # e.g. 40.0: keep only cells whose live-to-fixed registration distance is <= 40 px

print("repository root:", ROOT)
for name, path in INPUTS.items():
    print(f"{'OK     ' if path.exists() else 'MISSING'} {name}: {path.relative_to(ROOT)}")
assert all(p.exists() for p in INPUTS.values()), "missing input(s): run the cell cycle score notebook first for the labels"
assert SCRIPT.exists()

## 3. Run the analysis
This streams the log and overwrites `reports/ops0077_phase_predicts_cc_phase/`.

In [ ]:
cmd = [sys.executable, "-W", "ignore", str(SCRIPT), str(ROOT)]
if QUICK: cmd.append("--quick")
if INCLUDE_MASK_MORPHOLOGY: cmd.append("--include-mask-morphology")
if MAX_REG_DIST is not None: cmd += ["--max-reg-dist", str(MAX_REG_DIST)]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "analysis failed, see the log above"

## 4. Performance summary

In [ ]:
import pandas as pd
summary = json.load(open(OUT / "summary.json"))
print("best model:", summary["best_model"], "| features:", summary["n_features"], summary["feature_groups"],
      "| labelled cells:", summary["n_labelled"])
tab = pd.DataFrame(summary["results"]).T
tab = tab[[c for c in ["n", "mae_turns", "median_abs_err", "within_0p10", "acc_4bins", "acc_8bins", "circ_agreement"] if c in tab]]
tab.round(3)

**Reading the table.** `mae_turns` is the mean absolute circular error in units of one full cycle: a random guess gives 0.25, a perfect prediction 0. `within_0p10` is the fraction of cells predicted within 10% of a cycle (random: 0.20). `acc_4bins` and `acc_8bins` are accuracies after cutting the loop into 4 or 8 equal phase bins (random: 0.25 and 0.125). `circ_agreement` is 1 for a perfect prediction and 0 for an unrelated one. The `baseline_*` rows show what cell size alone, or registration distance and local cell number alone, would achieve. The `cp_subset` rows compare the models with and without the CellProfiler texture features on the same cells.

## 5. QC figures

In [ ]:
from IPython.display import Image, display
for f, cap in [("qc_pred_vs_true.png", "Predicted vs true phase, error distribution"),
               ("qc_error_by_phase.png", "Error along the cycle, and confusion between phase bins"),
               ("qc_error_drivers.png", "Confidence, registration distance, well, label quality, cell number"),
               ("qc_model_comparison.png", "Models vs baselines, tile-grouped and leave-one-well-out CV"),
               ("qc_profiles_true_vs_predicted.png", "DNA and marker profiles along the true vs predicted phase"),
               ("qc_feature_groups_and_top_features.png", "Feature groups and top single features")]:
    print(cap); display(Image(filename=str(OUT / f)))

## 6. Output files

| file | content |
|---|---|
| `phase_predicted_cc_phase.csv` | per cell: `pred_phase`, `pred_sin`, `pred_cos`, `pred_confidence`, `error_turns`, QC covariates and the label |
| `summary.json` | all metrics |
| `feature_table.csv`, `feature_importance.csv` | which features were used, their group and association with the phase |
| `ablation_by_group.csv` | performance with each feature group alone or left out |
| `error_by_*.csv` | error broken down by true phase, registration distance and well |

**`pred_phase` / `pred_is_out_of_fold`.** For cells with a label the prediction is out-of-fold (the model never saw that tile during training), so it is an honest estimate. For cells without a label (no reliable DNA measurement) it comes from a model trained on all labelled cells.

**Using the prediction.** The phase is circular, so compare phases with the circular difference `((a - b + 0.5) % 1) - 0.5`, or use `sin` and `cos`, never the raw 0-1 difference.